## Load EUR/USD exchange rate

In [1]:
import yfinance as yf

eur_usd = yf.download('EURUSD=X', start='1970-01-01')
brent = yf.download('BZ=F', start='1970-01-01')

print(eur_usd.index.min(), eur_usd.index.max())
print(brent.index.min(), brent.index.max())

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

2003-12-01 00:00:00 2026-08-11 00:00:00
2007-07-30 00:00:00 2026-08-11 00:00:00


## Connect to PostgreSQL and create raw fact tables

Following the same pattern as the Macro-Recession Monitor project: raw data lives in
PostgreSQL so it can be consumed by Power BI, Streamlit, and downstream notebooks from a
single source of truth.

In [2]:
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

load_dotenv()

user = os.getenv('DB_USER')
password = os.getenv('DB_PASSWORD')
host = os.getenv('DB_HOST')
port = os.getenv('DB_PORT')
dbname = os.getenv('DB_NAME')

engine = create_engine(f"postgresql://{user}:{password}@{host}:{port}/{dbname}")

In [3]:
eur_usd.head()

Price,Close,High,Low,Open,Volume
Ticker,EURUSD=X,EURUSD=X,EURUSD=X,EURUSD=X,EURUSD=X
Date,,,,,
2003-12-01,1.196501,1.204007,1.194401,1.203398,0
2003-12-02,1.208897,1.210903,1.194600,1.196101,0
2003-12-03,1.212298,1.213003,1.207700,1.209000,0
2003-12-04,1.208094,1.214403,1.204398,1.212004,0
2003-12-05,1.218695,1.219096,1.206593,1.207802,0


In [4]:
eur_usd.head()
eur_usd.columns

MultiIndex([( 'Close', 'EURUSD=X'),
            (  'High', 'EURUSD=X'),
            (   'Low', 'EURUSD=X'),
            (  'Open', 'EURUSD=X'),
            ('Volume', 'EURUSD=X')],
           names=['Price', 'Ticker'])

In [5]:
eur_usd_clean = eur_usd['Close'].reset_index()
eur_usd_clean.columns = ['obs_date', 'close_rate']

brent_clean = brent['Close'].reset_index()
brent_clean.columns = ['obs_date', 'close_price']

In [6]:
eur_usd_clean.head()

,obs_date,close_rate
0,2003-12-01,1.196501
1,2003-12-02,1.208897
2,2003-12-03,1.212298
3,2003-12-04,1.208094
4,2003-12-05,1.218695


In [7]:
eur_usd_clean.dtypes

obs_date      datetime64[s]
close_rate          float64
dtype: object

In [8]:
eur_usd_clean.to_sql('fact_eur_usd_rates', engine, if_exists='replace', index=False)
brent_clean.to_sql('fact_brent_prices', engine, if_exists='replace', index=False)

737

In [9]:
import pandas as pd

pd.read_sql("SELECT COUNT(*) FROM fact_eur_usd_rates", engine)

,count
0,5888


In [10]:
pd.read_sql("SELECT COUNT(*) FROM fact_brent_prices", engine)


,count
0,4737


## Load Irish pump prices (EU Weekly Oil Bulletin)

Official historical dataset from the European Commission, weekly, from 2005 onward.
Source: https://energy.ec.europa.eu/document/download/906e60ca-8b6a-44e7-8589-652854d2fd3f_en?filename=Weekly_Oil_Bulletin_Prices_History_maticni_4web.xlsx

Note: prices are typically published in EUR per 1,000 litres — will confirm exact units
and structure once the file is loaded, and convert to per-litre if needed for readability.

In [11]:
import requests

url = "https://energy.ec.europa.eu/document/download/906e60ca-8b6a-44e7-8589-652854d2fd3f_en?filename=Weekly_Oil_Bulletin_Prices_History_maticni_4web.xlsx"
response = requests.get(url)

with open('../data/raw/weekly_oil_bulletin_history.xlsx', 'wb') as f:
    f.write(response.content)

In [12]:
oil_bulletin_raw = pd.read_excel('../data/raw/weekly_oil_bulletin_history.xlsx', sheet_name=None)
oil_bulletin_raw.keys()

dict_keys(['Prices with taxes', 'Prices wo taxes', 'Consumption', 'VAT', 'Excise duties', 'Excise duties - components', 'Other Indirect Taxes'])

In [13]:
oil_bulletin_raw['Prices with taxes'].head(15)

,Consumer prices of petroleum products inclusive of duties and taxes,CTR,EU_price_with_tax_euro95,EU_price_with_tax_diesel,EU_price_with_tax_heating_oil,EU_price_with_tax_fuel_oil_1,EU_price_with_tax_fuel_oil_2,EU_price_with_tax_LPG,CTR.1,EUR_price_with_tax_euro95,...,SK_price_with_tax_fuel_oil_2,SK_price_with_tax_LPG,CTR.29,UK_exchange_rate,UK_price_with_tax_euro95,UK_price_with_tax_diesel,UK_price_with_tax_heating_oil,UK_price_with_tax_fuel_oil_1,UK_price_with_tax_fuel_oil_2,UK_price_with_tax_LPG
0,NaN,NaN,Euro-super 95 (I),Gas oil automobile Automotive gas oil Dieselkr...,Gas oil de chauffage Heating gas oil Heizöl (II),Fuel oil - Schweres Heizöl (III) Soufre,Fuel oil -Schweres Heizöl (III) Soufre > 1% S...,GPL pour moteur LPG motor fuel,NaN,Euro-super 95 (I),...,Fuel oil -Schweres Heizöl (III) Soufre > 1% S...,GPL pour moteur LPG motor fuel,NaN,NaN,Euro-super 95 (I),Gas oil automobile Automotive gas oil Dieselkr...,Gas oil de chauffage Heating gas oil Heizöl (II),Fuel oil - Schweres Heizöl (III) Soufre,Fuel oil -Schweres Heizöl (III) Soufre > 1% S...,GPL pour moteur LPG motor fuel
1,Date,NaN,1000 l,1000 l,1000 l,t,t,1000 l,NaN,1000 l,...,t,1000 l,NaN,NaN,1000 l,1000 l,1000 l,t,t,1000 l
2,2026-08-03 00:00:00,EU_,1952.243885,2043.111462,1443.108295,705.331549,540.553889,847.814879,EUR_,2005.586158,...,NaN,777,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-07-27 00:00:00,EU_,1940.488647,2009.471407,1448.426551,694.80212,573.154701,848.923378,EUR_,1991.939958,...,NaN,778,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-07-20 00:00:00,EU_,1907.712603,1926.45955,1411.721922,659.599275,531.192931,850.837588,EUR_,1958.931494,...,NaN,780,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2026-07-13 00:00:00,EU_,1851.0186,1822.917737,1309.083295,682.546922,491.665519,855.611785,EUR_,1905.706446,...,NaN,780,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2026-07-06 00:00:00,EU_,1814.340556,1766.174178,1229.85874,713.410299,469.644341,865.213075,EUR_,1864.351735,...,NaN,828,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2026-06-29 00:00:00,EU_,1756.555489,1715.77878,1205.367759,692.027446,490.011336,868.758918,EUR_,1806.644451,...,NaN,828,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2026-06-22 00:00:00,EU_,1761.454216,1730.150801,1202.394175,762.742891,501.026771,877.861379,EUR_,1814.762622,...,NaN,830,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2026-06-15 00:00:00,EU_,1804.705031,1804.387071,1272.764594,763.723423,574.199976,893.202017,EUR_,1856.501886,...,NaN,832,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
oil_bulletin_raw['Excise duties'].head(15)

,Unnamed: 0,Unnamed: 1,Excises - in national currency per unit (L or t),Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7
0,CTR,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,Since:,Euro-super 95 (I),Gas oil automobile Automotive gas oil Dieselkr...,Gas oil de chauffage Heating gas oil Heizöl (II),Fuel oil - Schweres Heizöl (III) Soufre,Fuel oil -Schweres Heizöl (III) Soufre > 1% S...,GPL pour moteur LPG motor fuel
2,NaN,NaN,1000 l,1000 l,1000 l,t,t,1000 l
3,AT_,2026-08-03 00:00:00,463,378,NaN,NaN,NaN,NaN
4,NaN,2026-07-06 00:00:00,474,389,NaN,NaN,NaN,NaN
5,NaN,2026-05-01 00:00:00,462,377,NaN,NaN,NaN,NaN
6,NaN,2026-04-01 00:00:00,432,347,NaN,NaN,NaN,NaN
7,NaN,2013-06-10 00:00:00,482,397,NaN,NaN,NaN,NaN
8,NaN,2013-06-01 00:00:00,482,397,NaN,NaN,NaN,NaN
9,NaN,2011-01-03 00:00:00,515,425,NaN,NaN,NaN,NaN


## Load Excise Duties (Ireland)

Historical excise rate changes for Ireland, parsed from the "Excise duties" sheet of the EU
Weekly Oil Bulletin. Kept in raw form (including rows where only heating gas oil changed,
not petrol/diesel) — will filter for petrol/diesel-specific event study later if needed.

In [15]:
excise_raw = pd.read_excel(
    '../data/raw/weekly_oil_bulletin_history.xlsx',
    sheet_name='Excise duties',
    header=None
)

excise_raw.head()

,0,1,2,3,4,5,6,7
0,NaN,NaN,Excises - in national currency per unit (L or t),NaN,NaN,NaN,NaN,NaN
1,CTR,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Since:,Euro-super 95 (I),Gas oil automobile Automotive gas oil Dieselkr...,Gas oil de chauffage Heating gas oil Heizöl (II),Fuel oil - Schweres Heizöl (III) Soufre,Fuel oil -Schweres Heizöl (III) Soufre > 1% S...,GPL pour moteur LPG motor fuel
3,NaN,NaN,1000 l,1000 l,1000 l,t,t,1000 l
4,AT_,2026-08-03 00:00:00,463,378,NaN,NaN,NaN,NaN


In [16]:
excise_data = excise_raw.iloc[4:].copy()
excise_data.columns = ['country', 'effective_date', 'petrol_excise', 'diesel_excise', 'heating_gas_oil', 'fuel_oil', 'fuel_oil_high_sulphur', 'lpg']

In [17]:
excise_data['country'] = excise_data['country'].ffill()

In [18]:
excise_ie = excise_data[excise_data['country'] == 'IE_'].copy()

In [19]:
excise_ie.head(15)

,country,effective_date,petrol_excise,diesel_excise,heating_gas_oil,fuel_oil,fuel_oil_high_sulphur,lpg
476,IE_,2026-03-25 00:00:00,584.18,453.15,193.06,NaN,NaN,NaN
477,IE_,2024-10-09 00:00:00,688.78,595.68,199.17,NaN,NaN,NaN
478,IE_,2024-08-01 00:00:00,671.43,575.61,199.17,NaN,NaN,NaN
479,IE_,2024-05-01 00:00:00,NaN,NaN,184.3,NaN,NaN,NaN
480,IE_,2024-04-01 00:00:00,638.91,551.22,163.96,197.64,NaN,NaN
481,IE_,2023-10-11 00:00:00,606.39,526.83,NaN,NaN,NaN,NaN
482,IE_,2023-09-01 00:00:00,589.03,506.75,149.09,NaN,NaN,NaN
483,IE_,2023-06-01 00:00:00,532.12,466.1,140.28,NaN,NaN,NaN
484,IE_,2023-05-01 00:00:00,NaN,NaN,131.47,173.26,NaN,NaN
485,IE_,2022-10-12 00:00:00,483.34,425.45,NaN,NaN,NaN,NaN


In [20]:
excise_ie.shape

(38, 8)

In [21]:
excise_ie.to_sql('fact_excise_rates', engine, if_exists='replace', index=False)

38

In [22]:
pd.read_sql("SELECT COUNT(*) FROM fact_excise_rates", engine)

,count
0,38


## Load Pump Prices with Taxes (Ireland)

Retail pump prices (petrol, diesel) inclusive of all taxes, parsed from the "Prices with
taxes" sheet of the EU Weekly Oil Bulletin. Unlike the Excise duties sheet, this one is in
a wide "grid" format — repeating blocks of columns per country (product prices), with row 0
holding product names and row 1 holding units. Ireland's block position is fixed across all
rows, but not fixed across the sheet layout itself, so the IE_ column index needs to be
located programmatically rather than hardcoded by trial and error.

In [23]:
prices_raw = pd.read_excel(
    '../data/raw/weekly_oil_bulletin_history.xlsx',
    sheet_name='Prices with taxes',
    header=0
)

prices_raw.head(15)

,Consumer prices of petroleum products inclusive of duties and taxes,CTR,EU_price_with_tax_euro95,EU_price_with_tax_diesel,EU_price_with_tax_heating_oil,EU_price_with_tax_fuel_oil_1,EU_price_with_tax_fuel_oil_2,EU_price_with_tax_LPG,CTR.1,EUR_price_with_tax_euro95,...,SK_price_with_tax_fuel_oil_2,SK_price_with_tax_LPG,CTR.29,UK_exchange_rate,UK_price_with_tax_euro95,UK_price_with_tax_diesel,UK_price_with_tax_heating_oil,UK_price_with_tax_fuel_oil_1,UK_price_with_tax_fuel_oil_2,UK_price_with_tax_LPG
0,NaN,NaN,Euro-super 95 (I),Gas oil automobile Automotive gas oil Dieselkr...,Gas oil de chauffage Heating gas oil Heizöl (II),Fuel oil - Schweres Heizöl (III) Soufre,Fuel oil -Schweres Heizöl (III) Soufre > 1% S...,GPL pour moteur LPG motor fuel,NaN,Euro-super 95 (I),...,Fuel oil -Schweres Heizöl (III) Soufre > 1% S...,GPL pour moteur LPG motor fuel,NaN,NaN,Euro-super 95 (I),Gas oil automobile Automotive gas oil Dieselkr...,Gas oil de chauffage Heating gas oil Heizöl (II),Fuel oil - Schweres Heizöl (III) Soufre,Fuel oil -Schweres Heizöl (III) Soufre > 1% S...,GPL pour moteur LPG motor fuel
1,Date,NaN,1000 l,1000 l,1000 l,t,t,1000 l,NaN,1000 l,...,t,1000 l,NaN,NaN,1000 l,1000 l,1000 l,t,t,1000 l
2,2026-08-03 00:00:00,EU_,1952.243885,2043.111462,1443.108295,705.331549,540.553889,847.814879,EUR_,2005.586158,...,NaN,777,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-07-27 00:00:00,EU_,1940.488647,2009.471407,1448.426551,694.80212,573.154701,848.923378,EUR_,1991.939958,...,NaN,778,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-07-20 00:00:00,EU_,1907.712603,1926.45955,1411.721922,659.599275,531.192931,850.837588,EUR_,1958.931494,...,NaN,780,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2026-07-13 00:00:00,EU_,1851.0186,1822.917737,1309.083295,682.546922,491.665519,855.611785,EUR_,1905.706446,...,NaN,780,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2026-07-06 00:00:00,EU_,1814.340556,1766.174178,1229.85874,713.410299,469.644341,865.213075,EUR_,1864.351735,...,NaN,828,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2026-06-29 00:00:00,EU_,1756.555489,1715.77878,1205.367759,692.027446,490.011336,868.758918,EUR_,1806.644451,...,NaN,828,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2026-06-22 00:00:00,EU_,1761.454216,1730.150801,1202.394175,762.742891,501.026771,877.861379,EUR_,1814.762622,...,NaN,830,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2026-06-15 00:00:00,EU_,1804.705031,1804.387071,1272.764594,763.723423,574.199976,893.202017,EUR_,1856.501886,...,NaN,832,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
date_col = prices_raw.columns[0]
ie_cols = [col for col in prices_raw.columns if col.startswith('IE_')]

prices_data = prices_raw[[date_col] + ie_cols].copy()
prices_data.head(15)

,Consumer prices of petroleum products inclusive of duties and taxes,IE_price_with_tax_euro95,IE_price_with_tax_diesel,IE_price_with_tax_heating_oil,IE_price_with_tax_fuel_oil_1,IE_price_with_tax_fuel_oil_2,IE_price_with_tax_LPG
0,NaN,Euro-super 95 (I),Gas oil automobile Automotive gas oil Dieselkr...,Gas oil de chauffage Heating gas oil Heizöl (II),Fuel oil - Schweres Heizöl (III) Soufre,Fuel oil -Schweres Heizöl (III) Soufre > 1% S...,GPL pour moteur LPG motor fuel
1,Date,1000 l,1000 l,1000 l,t,t,1000 l
2,2026-08-03 00:00:00,1817.3,1845.6,1360.95,NaN,NaN,NaN
3,2026-07-27 00:00:00,1757.7,1751.7,1324.5,NaN,NaN,NaN
4,2026-07-20 00:00:00,1720.2,1695,1262.95,NaN,NaN,NaN
5,2026-07-13 00:00:00,1712.5,1689.3,1113.25,NaN,NaN,NaN
6,2026-07-06 00:00:00,1729.8,1712.7,1151.6,NaN,NaN,NaN
7,2026-06-29 00:00:00,1762.2,1759.2,1194.55,NaN,NaN,NaN
8,2026-06-22 00:00:00,1793.9,1807.3,1228,985.48,NaN,NaN
9,2026-06-15 00:00:00,1813.9,1844.4,1275,1037.8,NaN,NaN


In [25]:
list(prices_raw.columns)

['Consumer prices of petroleum products inclusive of duties and taxes',
 'CTR',
 'EU_price_with_tax_euro95',
 'EU_price_with_tax_diesel',
 'EU_price_with_tax_heating_oil',
 'EU_price_with_tax_fuel_oil_1',
 'EU_price_with_tax_fuel_oil_2',
 'EU_price_with_tax_LPG',
 'CTR.1',
 'EUR_price_with_tax_euro95',
 'EUR_price_with_tax_diesel',
 'EUR_price_with_tax_heating_oil',
 'EUR_price_with_tax_fuel_oil_1',
 'EUR_price_with_tax_fuel_oil_2',
 'EUR_price_with_tax_LPG',
 'CTR.2',
 'AT_price_with_tax_euro95',
 'AT_price_with_tax_diesel',
 'AT_price_with_tax_heating_oil',
 'AT_price_with_tax_fuel_oil_1',
 'AT_price_with_tax_fuel_oil_2',
 'AT_price_with_tax_LPG',
 'CTR.3',
 'BE_price_with_tax_euro95',
 'BE_price_with_tax_diesel',
 'BE_price_with_tax_heating_oil',
 'BE_price_with_tax_fuel_oil_1',
 'BE_price_with_tax_fuel_oil_2',
 'BE_price_with_tax_LPG',
 'CTR.4',
 'BG_exchange_rate',
 'BG_price_with_tax_euro95',
 'BG_price_with_tax_diesel',
 'BG_price_with_tax_heating_oil',
 'BG_price_with_tax_fuel_

In [27]:
prices_data = prices_data.iloc[2:].reset_index(drop=True)

prices_data.columns = ['week_date', 'petrol_price', 'diesel_price', 'heating_oil_price', 'fuel_oil_1', 'fuel_oil_2', 'lpg_price']

prices_ie = prices_data[['week_date', 'petrol_price', 'diesel_price']].copy()

In [28]:
prices_ie.head(10)

,week_date,petrol_price,diesel_price
0,2026-08-03 00:00:00,1817.3,1845.6
1,2026-07-27 00:00:00,1757.7,1751.7
2,2026-07-20 00:00:00,1720.2,1695
3,2026-07-13 00:00:00,1712.5,1689.3
4,2026-07-06 00:00:00,1729.8,1712.7
5,2026-06-29 00:00:00,1762.2,1759.2
6,2026-06-22 00:00:00,1793.9,1807.3
7,2026-06-15 00:00:00,1813.9,1844.4
8,2026-06-08 00:00:00,1838.2,1896.7
9,2026-06-01 00:00:00,1838.8,1922.2


In [32]:
prices_ie[prices_ie['week_date'].astype(str).str.contains('Notes', na=False)]

,week_date,petrol_price,diesel_price
1080,Notes:,NaN,NaN


In [33]:
prices_ie = prices_ie[pd.to_datetime(prices_ie['week_date'], errors='coerce').notna()].copy()
prices_ie['week_date'] = pd.to_datetime(prices_ie['week_date'])

In [34]:
prices_ie.shape

(1078, 3)

In [35]:
prices_ie.dtypes

week_date       datetime64[us]
petrol_price            object
diesel_price            object
dtype: object

In [36]:
prices_ie['petrol_price'].apply(type).value_counts()

petrol_price
<class 'int'>      638
<class 'float'>    440
Name: count, dtype: int64

In [37]:
prices_ie['petrol_price'] = pd.to_numeric(prices_ie['petrol_price'], errors='coerce')
prices_ie['diesel_price'] = pd.to_numeric(prices_ie['diesel_price'], errors='coerce')

In [38]:
prices_ie.dtypes

week_date       datetime64[us]
petrol_price           float64
diesel_price           float64
dtype: object

## Load Pump Prices without Taxes (Ireland)

Same "Prices wo taxes" sheet, same grid structure and parsing logic as the taxed version
above. Pre-tax prices let us later isolate how much of the tax component (excise, and
implicitly VAT) contributes to price moves, versus the pure market/refining price — useful
for a cleaner read on the pass-through relationship with Brent, separate from tax policy
changes happening in the same window.

In [39]:
prices_wo_tax_raw = pd.read_excel(
    '../data/raw/weekly_oil_bulletin_history.xlsx',
    sheet_name='Prices wo taxes',
    header=0
)

date_col = prices_wo_tax_raw.columns[0]
ie_cols = [col for col in prices_wo_tax_raw.columns if col.startswith('IE_')]

prices_wo_tax_data = prices_wo_tax_raw[[date_col] + ie_cols].copy()
prices_wo_tax_data = prices_wo_tax_data.iloc[2:].reset_index(drop=True)
prices_wo_tax_data.columns = ['week_date', 'petrol_price_pretax', 'diesel_price_pretax', 'heating_oil_price_pretax', 'fuel_oil_1_pretax', 'fuel_oil_2_pretax', 'lpg_price_pretax']

prices_wo_tax_ie = prices_wo_tax_data[['week_date', 'petrol_price_pretax', 'diesel_price_pretax']].copy()

prices_wo_tax_ie = prices_wo_tax_ie[pd.to_datetime(prices_wo_tax_ie['week_date'], errors='coerce').notna()].copy()
prices_wo_tax_ie['week_date'] = pd.to_datetime(prices_wo_tax_ie['week_date'])
prices_wo_tax_ie['petrol_price_pretax'] = pd.to_numeric(prices_wo_tax_ie['petrol_price_pretax'], errors='coerce')
prices_wo_tax_ie['diesel_price_pretax'] = pd.to_numeric(prices_wo_tax_ie['diesel_price_pretax'], errors='coerce')

In [40]:
prices_wo_tax_ie.dtypes

week_date              datetime64[us]
petrol_price_pretax           float64
diesel_price_pretax           float64
dtype: object

In [41]:
prices_wo_tax_ie.shape

(1078, 3)

In [42]:
prices_wo_tax_ie['petrol_price_pretax'].isna().sum()

np.int64(0)

In [43]:
prices_wo_tax_ie['diesel_price_pretax'].isna().sum()

np.int64(0)

In [44]:
prices_wo_tax_ie.head()

,week_date,petrol_price_pretax,diesel_price_pretax
0,2026-08-03,873.299675,1027.337805
1,2026-07-27,824.844390,950.996341
2,2026-07-20,794.356585,904.898780
3,2026-07-13,788.096423,900.264634
4,2026-07-06,802.161463,919.289024


In [47]:
prices_ie.to_sql('fact_pump_prices', engine, if_exists='replace', index=False)
prices_wo_tax_ie.to_sql('fact_pump_prices_pretax', engine, if_exists='replace', index=False)

78

In [48]:
pd.read_sql("SELECT COUNT(*) FROM fact_pump_prices", engine)

,count
0,1078


In [49]:
pd.read_sql("SELECT COUNT(*) FROM fact_pump_prices_pretax", engine)

,count
0,1078
